In [1]:
import pandas as pd
import re
import os
import json

# Load existing career skills
career_df = pd.read_csv(r'C:\CareerShieldAI\datasets\career_skills.csv')
print(f"Existing career roles: {len(career_df)}")

# Load Naukri datasets
naukri1 = pd.read_csv(r'C:\CareerShieldAI\datasets\NaukriData_Data Science.csv',
                       encoding='utf-8', on_bad_lines='skip')
naukri2 = pd.read_csv(r'C:\CareerShieldAI\datasets\Naukri_Data_Scientist_and_Data_Analytics_Jobs_Data.csv',
                       encoding='latin1', on_bad_lines='skip')
naukri3 = pd.read_csv(r'C:\CareerShieldAI\datasets\NaukriData_data analytics.csv',
                       encoding='utf-8', on_bad_lines='skip')
salary_df = pd.read_csv(r'C:\CareerShieldAI\datasets\job_market_india.csv',
                         encoding='utf-8', on_bad_lines='skip')

print(f"Naukri DS:        {len(naukri1)} rows")
print(f"Naukri DS+Ana:    {len(naukri2)} rows")
print(f"Naukri Analytics: {len(naukri3)} rows")
print(f"Job Market India: {len(salary_df)} rows")
print("All datasets loaded.")

Existing career roles: 4643
Naukri DS:        47191 rows
Naukri DS+Ana:    19649 rows
Naukri Analytics: 20064 rows
Job Market India: 835 rows
All datasets loaded.


In [2]:
# Fix Naukri skills - they are concatenated without separator
# "Text miningData scienceMachine learning" → "Text mining, Data science, Machine learning"
def parse_naukri_skills(skills_text):
    if not isinstance(skills_text, str) or skills_text.strip() == '':
        return ''
    # Split on capital letters (each skill starts with capital)
    skills = re.sub(r'([a-z])([A-Z])', r'\1, \2', skills_text)
    skills = re.sub(r'\s+', ' ', skills).strip()
    return skills

# Standardize all 3 Naukri files to same format
def standardize_naukri(df, title_col, skills_col):
    result = pd.DataFrame()
    result['job_title'] = df[title_col].fillna('')
    result['skills']    = df[skills_col].apply(parse_naukri_skills)
    result['source']    = 'naukri'
    return result[result['job_title'] != '']

naukri1_clean = standardize_naukri(naukri1, 'Job_Titles', 'Skills')
naukri2_clean = standardize_naukri(naukri2, 'Job Titles', 'Skills')
naukri3_clean = standardize_naukri(naukri3, 'Job_Titles', 'Skills')

print(f"Naukri1 cleaned: {len(naukri1_clean)}")
print(f"Naukri2 cleaned: {len(naukri2_clean)}")
print(f"Naukri3 cleaned: {len(naukri3_clean)}")
print(f"\nSample parsed skills:")
print(naukri1_clean['skills'].iloc[0][:150])

Naukri1 cleaned: 33633
Naukri2 cleaned: 19522
Naukri3 cleaned: 20041

Sample parsed skills:
Text mining, Career developmentdata science, Finance, Machine learning, Data mining, Stakeholder management, SQL


In [3]:
# Merge all datasets into one master career dataset
career_df['source'] = 'original'

master_df = pd.concat([
    career_df[['job_title', 'skills', 'source']],
    naukri1_clean,
    naukri2_clean,
    naukri3_clean
], ignore_index=True)

# Remove duplicates and empty skills
master_df = master_df[master_df['skills'].str.strip() != '']
master_df.drop_duplicates(subset=['job_title', 'skills'], inplace=True)
master_df.reset_index(drop=True, inplace=True)

print(f"Original roles:  4,643")
print(f"After merge:     {len(master_df)}")
print(f"\nSource breakdown:")
print(master_df['source'].value_counts())

# Save
master_df.to_csv(r'C:\CareerShieldAI\datasets\master_career_skills.csv', index=False)
print("\nSaved to datasets/master_career_skills.csv")

Original roles:  4,643
After merge:     52226

Source breakdown:
source
naukri      47583
original     4643
Name: count, dtype: int64

Saved to datasets/master_career_skills.csv


In [4]:
# Build complete salary & demand table for Indian market
salary_demand_data = [

    # Data Science & AI
    {"role": "Data Scientist",              "min_lpa": 8,  "max_lpa": 25, "avg_lpa": 14, "demand": "Very High", "growth": "Excellent"},
    {"role": "Machine Learning Engineer",   "min_lpa": 10, "max_lpa": 30, "avg_lpa": 18, "demand": "Very High", "growth": "Excellent"},
    {"role": "AI Engineer",                 "min_lpa": 12, "max_lpa": 35, "avg_lpa": 20, "demand": "Very High", "growth": "Excellent"},
    {"role": "Deep Learning Engineer",      "min_lpa": 12, "max_lpa": 35, "avg_lpa": 22, "demand": "High",      "growth": "Excellent"},
    {"role": "NLP Engineer",                "min_lpa": 10, "max_lpa": 30, "avg_lpa": 18, "demand": "High",      "growth": "Excellent"},
    {"role": "Computer Vision Engineer",    "min_lpa": 10, "max_lpa": 28, "avg_lpa": 17, "demand": "High",      "growth": "Very Good"},
    {"role": "MLOps Engineer",              "min_lpa": 12, "max_lpa": 32, "avg_lpa": 20, "demand": "Very High", "growth": "Excellent"},
    {"role": "Data Analyst",                "min_lpa": 4,  "max_lpa": 15, "avg_lpa": 7,  "demand": "Very High", "growth": "Good"},
    {"role": "Business Analyst",            "min_lpa": 5,  "max_lpa": 18, "avg_lpa": 9,  "demand": "High",      "growth": "Good"},
    {"role": "Data Engineer",               "min_lpa": 8,  "max_lpa": 25, "avg_lpa": 14, "demand": "Very High", "growth": "Excellent"},
    {"role": "Analytics Engineer",          "min_lpa": 8,  "max_lpa": 22, "avg_lpa": 13, "demand": "High",      "growth": "Very Good"},
    {"role": "Generative AI Engineer",      "min_lpa": 15, "max_lpa": 50, "avg_lpa": 28, "demand": "Very High", "growth": "Excellent"},
    {"role": "Prompt Engineer",             "min_lpa": 10, "max_lpa": 30, "avg_lpa": 18, "demand": "Very High", "growth": "Excellent"},
    {"role": "Research Scientist",          "min_lpa": 15, "max_lpa": 40, "avg_lpa": 25, "demand": "High",      "growth": "Excellent"},

    # Software Development
    {"role": "Software Engineer",           "min_lpa": 5,  "max_lpa": 20, "avg_lpa": 10, "demand": "Very High", "growth": "Good"},
    {"role": "Senior Software Engineer",    "min_lpa": 12, "max_lpa": 35, "avg_lpa": 20, "demand": "Very High", "growth": "Good"},
    {"role": "Full Stack Developer",        "min_lpa": 5,  "max_lpa": 22, "avg_lpa": 11, "demand": "Very High", "growth": "Very Good"},
    {"role": "Backend Developer",           "min_lpa": 5,  "max_lpa": 20, "avg_lpa": 10, "demand": "High",      "growth": "Good"},
    {"role": "Frontend Developer",          "min_lpa": 4,  "max_lpa": 18, "avg_lpa": 8,  "demand": "High",      "growth": "Good"},
    {"role": "Python Developer",            "min_lpa": 5,  "max_lpa": 22, "avg_lpa": 11, "demand": "Very High", "growth": "Very Good"},
    {"role": "Java Developer",              "min_lpa": 5,  "max_lpa": 20, "avg_lpa": 10, "demand": "High",      "growth": "Good"},
    {"role": "React Developer",             "min_lpa": 4,  "max_lpa": 18, "avg_lpa": 9,  "demand": "High",      "growth": "Good"},
    {"role": "Node.js Developer",           "min_lpa": 4,  "max_lpa": 18, "avg_lpa": 8,  "demand": "High",      "growth": "Good"},
    {"role": "Mobile App Developer",        "min_lpa": 4,  "max_lpa": 18, "avg_lpa": 9,  "demand": "High",      "growth": "Good"},
    {"role": "Android Developer",           "min_lpa": 4,  "max_lpa": 18, "avg_lpa": 9,  "demand": "High",      "growth": "Good"},
    {"role": "iOS Developer",               "min_lpa": 5,  "max_lpa": 20, "avg_lpa": 10, "demand": "Medium",    "growth": "Good"},

    # Cloud & DevOps
    {"role": "Cloud Engineer",              "min_lpa": 8,  "max_lpa": 28, "avg_lpa": 16, "demand": "Very High", "growth": "Excellent"},
    {"role": "DevOps Engineer",             "min_lpa": 8,  "max_lpa": 28, "avg_lpa": 15, "demand": "Very High", "growth": "Excellent"},
    {"role": "AWS Solutions Architect",     "min_lpa": 12, "max_lpa": 40, "avg_lpa": 22, "demand": "Very High", "growth": "Excellent"},
    {"role": "Kubernetes Engineer",         "min_lpa": 10, "max_lpa": 30, "avg_lpa": 18, "demand": "High",      "growth": "Excellent"},
    {"role": "Site Reliability Engineer",   "min_lpa": 12, "max_lpa": 35, "avg_lpa": 20, "demand": "High",      "growth": "Very Good"},
    {"role": "Platform Engineer",           "min_lpa": 10, "max_lpa": 30, "avg_lpa": 18, "demand": "High",      "growth": "Very Good"},

    # Cybersecurity
    {"role": "Cybersecurity Analyst",       "min_lpa": 6,  "max_lpa": 20, "avg_lpa": 11, "demand": "Very High", "growth": "Excellent"},
    {"role": "Security Engineer",           "min_lpa": 8,  "max_lpa": 25, "avg_lpa": 14, "demand": "Very High", "growth": "Excellent"},
    {"role": "Penetration Tester",          "min_lpa": 6,  "max_lpa": 22, "avg_lpa": 12, "demand": "High",      "growth": "Very Good"},
    {"role": "Network Security Engineer",   "min_lpa": 6,  "max_lpa": 20, "avg_lpa": 11, "demand": "High",      "growth": "Good"},

    # Networking & Infrastructure
    {"role": "Network Engineer",            "min_lpa": 4,  "max_lpa": 15, "avg_lpa": 7,  "demand": "Medium",    "growth": "Stable"},
    {"role": "Network Administrator",       "min_lpa": 3,  "max_lpa": 12, "avg_lpa": 6,  "demand": "Medium",    "growth": "Stable"},
    {"role": "System Administrator",        "min_lpa": 3,  "max_lpa": 12, "avg_lpa": 6,  "demand": "Medium",    "growth": "Stable"},
    {"role": "Computer Network Specialist", "min_lpa": 4,  "max_lpa": 15, "avg_lpa": 8,  "demand": "Medium",    "growth": "Stable"},

    # Management
    {"role": "Product Manager",             "min_lpa": 12, "max_lpa": 40, "avg_lpa": 22, "demand": "High",      "growth": "Very Good"},
    {"role": "Project Manager",             "min_lpa": 8,  "max_lpa": 25, "avg_lpa": 14, "demand": "High",      "growth": "Good"},
    {"role": "IT Consultant",               "min_lpa": 8,  "max_lpa": 25, "avg_lpa": 14, "demand": "High",      "growth": "Good"},
    {"role": "Scrum Master",                "min_lpa": 8,  "max_lpa": 22, "avg_lpa": 13, "demand": "Medium",    "growth": "Good"},
    {"role": "Technical Lead",              "min_lpa": 15, "max_lpa": 40, "avg_lpa": 25, "demand": "High",      "growth": "Very Good"},
    {"role": "Engineering Manager",         "min_lpa": 20, "max_lpa": 60, "avg_lpa": 35, "demand": "High",      "growth": "Very Good"},

    # Finance & Accounting
    {"role": "Financial Analyst",           "min_lpa": 4,  "max_lpa": 15, "avg_lpa": 8,  "demand": "High",      "growth": "Good"},
    {"role": "Chartered Accountant",        "min_lpa": 6,  "max_lpa": 20, "avg_lpa": 10, "demand": "High",      "growth": "Good"},
    {"role": "Investment Analyst",          "min_lpa": 6,  "max_lpa": 20, "avg_lpa": 10, "demand": "Medium",    "growth": "Good"},
    {"role": "Risk Analyst",                "min_lpa": 5,  "max_lpa": 18, "avg_lpa": 9,  "demand": "High",      "growth": "Good"},
    {"role": "Credit Analyst",              "min_lpa": 4,  "max_lpa": 14, "avg_lpa": 7,  "demand": "Medium",    "growth": "Stable"},
    {"role": "Quantitative Analyst",        "min_lpa": 10, "max_lpa": 35, "avg_lpa": 20, "demand": "High",      "growth": "Very Good"},

    # HR & Operations
    {"role": "HR Manager",                  "min_lpa": 5,  "max_lpa": 18, "avg_lpa": 9,  "demand": "Medium",    "growth": "Stable"},
    {"role": "Talent Acquisition Specialist","min_lpa": 4, "max_lpa": 14, "avg_lpa": 7,  "demand": "Medium",    "growth": "Stable"},
    {"role": "Operations Manager",          "min_lpa": 6,  "max_lpa": 20, "avg_lpa": 11, "demand": "Medium",    "growth": "Good"},
    {"role": "HR Business Partner",         "min_lpa": 8,  "max_lpa": 22, "avg_lpa": 13, "demand": "Medium",    "growth": "Good"},

    # Design & Content
    {"role": "UI/UX Designer",              "min_lpa": 4,  "max_lpa": 18, "avg_lpa": 9,  "demand": "High",      "growth": "Good"},
    {"role": "Graphic Designer",            "min_lpa": 3,  "max_lpa": 10, "avg_lpa": 5,  "demand": "Medium",    "growth": "Stable"},
    {"role": "Content Writer",              "min_lpa": 2,  "max_lpa": 8,  "avg_lpa": 4,  "demand": "Medium",    "growth": "Stable"},
    {"role": "Digital Marketing Specialist","min_lpa": 3,  "max_lpa": 12, "avg_lpa": 6,  "demand": "High",      "growth": "Good"},
    {"role": "SEO Specialist",              "min_lpa": 3,  "max_lpa": 10, "avg_lpa": 5,  "demand": "Medium",    "growth": "Stable"},

    # Healthcare
    {"role": "Healthcare Analyst",          "min_lpa": 4,  "max_lpa": 14, "avg_lpa": 7,  "demand": "Medium",    "growth": "Good"},
    {"role": "Clinical Data Manager",       "min_lpa": 6,  "max_lpa": 18, "avg_lpa": 10, "demand": "Medium",    "growth": "Good"},

    # Emerging
    {"role": "Blockchain Developer",        "min_lpa": 8,  "max_lpa": 25, "avg_lpa": 15, "demand": "Medium",    "growth": "Good"},
    {"role": "AR/VR Developer",             "min_lpa": 6,  "max_lpa": 20, "avg_lpa": 12, "demand": "Medium",    "growth": "Very Good"},
    {"role": "IoT Engineer",                "min_lpa": 5,  "max_lpa": 18, "avg_lpa": 10, "demand": "Medium",    "growth": "Good"},
    {"role": "Robotics Engineer",           "min_lpa": 6,  "max_lpa": 22, "avg_lpa": 13, "demand": "Medium",    "growth": "Very Good"},
]

salary_demand_df = pd.DataFrame(salary_demand_data)
salary_demand_df.to_csv(r'C:\CareerShieldAI\datasets\salary_demand.csv', index=False)

print(f"Total roles in salary table: {len(salary_demand_df)}")
print(f"Demand breakdown:")
print(salary_demand_df['demand'].value_counts())
print("\nSaved to datasets/salary_demand.csv")

Total roles in salary table: 67
Demand breakdown:
demand
High         29
Medium       21
Very High    17
Name: count, dtype: int64

Saved to datasets/salary_demand.csv


In [5]:
# Build learning resources for 45+ skills
learning_resources = {
    "python":           {"youtube": "https://youtu.be/rfscVS0vtbw",  "course": "https://www.coursera.org/learn/python",                "book": "Automate the Boring Stuff with Python"},
    "java":             {"youtube": "https://youtu.be/eIrMbAQSU34",  "course": "https://www.udemy.com/course/java-the-complete-java-developer-course", "book": "Head First Java"},
    "javascript":       {"youtube": "https://youtu.be/W6NZfCO5SIk",  "course": "https://www.udemy.com/course/the-complete-javascript-course", "book": "Eloquent JavaScript"},
    "sql":              {"youtube": "https://youtu.be/HXV3zeQKqGY",  "course": "https://www.coursera.org/learn/sql-for-data-science",   "book": "Learning SQL by Alan Beaulieu"},
    "react":            {"youtube": "https://youtu.be/bMknfKXIFA8",  "course": "https://www.udemy.com/course/react-the-complete-guide-incl-redux", "book": "Learning React by Alex Banks"},
    "django":           {"youtube": "https://youtu.be/F5mRW0jo-U4",  "course": "https://www.udemy.com/course/python-and-django-full-stack-web-developer-bootcamp", "book": "Django for Beginners"},
    "flask":            {"youtube": "https://youtu.be/Z1RJmh_OqeA",  "course": "https://www.udemy.com/course/python-flask-for-beginners", "book": "Flask Web Development"},
    "node.js":          {"youtube": "https://youtu.be/TlB_eWDSMt4",  "course": "https://www.udemy.com/course/the-complete-nodejs-developer-course-2", "book": "Node.js Design Patterns"},
    "html":             {"youtube": "https://youtu.be/qz0aGYrrlhU",  "course": "https://www.coursera.org/learn/html-css-javascript-for-web-developers", "book": "HTML and CSS by Jon Duckett"},
    "css":              {"youtube": "https://youtu.be/yfoY53QXEnI",  "course": "https://www.udemy.com/course/css-the-complete-guide-incl-flexbox-grid-sass", "book": "CSS: The Definitive Guide"},
    "aws":              {"youtube": "https://youtu.be/k1RI5locZE4",  "course": "https://www.udemy.com/course/aws-certified-solutions-architect-associate-saa-c03", "book": "AWS Certified Solutions Architect Study Guide"},
    "azure":            {"youtube": "https://youtu.be/NKEFWyqJ5XA",  "course": "https://www.coursera.org/learn/microsoft-azure-fundamentals-az-900", "book": "Exam Ref AZ-900 Microsoft Azure Fundamentals"},
    "docker":           {"youtube": "https://youtu.be/3c-iBn73dDE",  "course": "https://www.udemy.com/course/docker-mastery",           "book": "Docker Deep Dive by Nigel Poulton"},
    "kubernetes":       {"youtube": "https://youtu.be/X48VuDVv0do",  "course": "https://www.udemy.com/course/certified-kubernetes-administrator-with-practice-tests", "book": "Kubernetes in Action"},
    "devops":           {"youtube": "https://youtu.be/0yWAtQ6wYNM",  "course": "https://www.coursera.org/professional-certificates/devops-and-software-engineering", "book": "The Phoenix Project"},
    "machine learning": {"youtube": "https://youtu.be/NWONeJKn6kc",  "course": "https://www.coursera.org/learn/machine-learning",       "book": "Hands-On ML with Scikit-Learn"},
    "deep learning":    {"youtube": "https://youtu.be/aircAruvnKk",  "course": "https://www.coursera.org/specializations/deep-learning", "book": "Deep Learning by Ian Goodfellow"},
    "tensorflow":       {"youtube": "https://youtu.be/tPYj3fFJGjk",  "course": "https://www.coursera.org/professional-certificates/tensorflow-in-practice", "book": "Learning TensorFlow"},
    "pytorch":          {"youtube": "https://youtu.be/Z_ikDlimN6A",  "course": "https://www.udemy.com/course/pytorch-for-deep-learning", "book": "Programming PyTorch for Deep Learning"},
    "pandas":           {"youtube": "https://youtu.be/vmEHCJofslg",  "course": "https://www.kaggle.com/learn/pandas",                   "book": "Python for Data Analysis"},
    "numpy":            {"youtube": "https://youtu.be/QUT1VHiLmmI",  "course": "https://www.kaggle.com/learn/intro-to-machine-learning", "book": "From Python to Numpy"},
    "data analysis":    {"youtube": "https://youtu.be/r-uOLxNrNk8",  "course": "https://www.coursera.org/professional-certificates/google-data-analytics", "book": "Storytelling with Data"},
    "nlp":              {"youtube": "https://youtu.be/X2vAabgKiuM",  "course": "https://www.coursera.org/specializations/natural-language-processing", "book": "Speech and Language Processing"},
    "power bi":         {"youtube": "https://youtu.be/AGrl-H87pRU",  "course": "https://www.udemy.com/course/microsoft-power-bi-up-running-with-power-bi-desktop", "book": "Beginning Power BI"},
    "tableau":          {"youtube": "https://youtu.be/aHaOIvR00So",  "course": "https://www.udemy.com/course/tableau10",                "book": "Learning Tableau"},
    "mongodb":          {"youtube": "https://youtu.be/-56x56UppqQ",  "course": "https://www.udemy.com/course/mongodb-the-complete-developers-guide", "book": "MongoDB: The Definitive Guide"},
    "postgresql":       {"youtube": "https://youtu.be/qw--VYLpxG4",  "course": "https://www.udemy.com/course/the-complete-python-postgresql-developer-course", "book": "PostgreSQL: Up and Running"},
    "mysql":            {"youtube": "https://youtu.be/7S_tz1z_5bA",  "course": "https://www.udemy.com/course/the-ultimate-mysql-bootcamp-go-from-sql-beginner-to-expert", "book": "Learning MySQL"},
    "cybersecurity":    {"youtube": "https://youtu.be/hXSFdwIOfnE",  "course": "https://www.coursera.org/professional-certificates/google-cybersecurity", "book": "The Web Application Hackers Handbook"},
    "network security": {"youtube": "https://youtu.be/2GNNjgus_5s",  "course": "https://www.coursera.org/learn/network-security-database-vulnerabilities", "book": "Network Security Essentials"},
    "project management":{"youtube": "https://youtu.be/FpgYdq1W1dk","course": "https://www.coursera.org/professional-certificates/google-project-management", "book": "PMBOK Guide"},
    "agile":            {"youtube": "https://youtu.be/Z9QbYZh1YXY",  "course": "https://www.coursera.org/learn/agile-development",      "book": "The Agile Samurai"},
    "scrum":            {"youtube": "https://youtu.be/2Vt7Ik8Ublw",  "course": "https://www.udemy.com/course/scrum-advanced",           "book": "Scrum by Jeff Sutherland"},
    "financial analysis":{"youtube": "https://youtu.be/7J9EM1G4mk4","course": "https://www.coursera.org/learn/financial-analysis",     "book": "Financial Statement Analysis"},
    "accounting":       {"youtube": "https://youtu.be/yYX4bvQSqbo",  "course": "https://www.coursera.org/learn/wharton-accounting",     "book": "Accounting Made Simple"},
    "excel":            {"youtube": "https://youtu.be/Vl0H-qTclOg",  "course": "https://www.udemy.com/course/microsoft-excel-2013-from-beginner-to-advanced-and-beyond", "book": "Excel Bible"},
    "cisco":            {"youtube": "https://youtu.be/rv3QK2UquxM",  "course": "https://www.udemy.com/course/complete-networking-fundamentals-course-ccna-start", "book": "CCNA 200-301 Official Cert Guide"},
    "networking":       {"youtube": "https://youtu.be/qiQR5rTSshw",  "course": "https://www.coursera.org/learn/computer-networking",    "book": "Computer Networks by Tanenbaum"},
    "linux":            {"youtube": "https://youtu.be/wBp0Rb-ZJak",  "course": "https://www.udemy.com/course/complete-linux-training-course-to-get-your-dream-it-job", "book": "The Linux Command Line"},
    "git":              {"youtube": "https://youtu.be/RGOj5yH7evk",  "course": "https://www.udemy.com/course/git-complete",             "book": "Pro Git by Scott Chacon"},
    "rest api":         {"youtube": "https://youtu.be/lsMQRaeKNDk",  "course": "https://www.udemy.com/course/rest-api-flask-and-python", "book": "RESTful Web APIs"},
    "microservices":    {"youtube": "https://youtu.be/lTAcCNbJ7KE",  "course": "https://www.udemy.com/course/microservices-with-node-js-and-react", "book": "Building Microservices"},
    "terraform":        {"youtube": "https://youtu.be/SLB_c_ayRMo",  "course": "https://www.udemy.com/course/terraform-beginner-to-advanced", "book": "Terraform: Up and Running"},
    "jenkins":          {"youtube": "https://youtu.be/LFDrDnKPOTg",  "course": "https://www.udemy.com/course/jenkins-from-zero-to-hero", "book": "Jenkins 2: Up and Running"},
    "blockchain":       {"youtube": "https://youtu.be/SSo_EIwHSd4",  "course": "https://www.coursera.org/learn/blockchain-basics",      "book": "Mastering Bitcoin"},
    "c++":              {"youtube": "https://youtu.be/vLnPwxZdW4Y",  "course": "https://www.udemy.com/course/beginning-c-plus-plus-programming", "book": "C++ Primer"},
    "system administration":{"youtube": "https://youtu.be/UCr04qIB7uc","course": "https://www.coursera.org/learn/system-administration-it-infrastructure-services", "book": "The Practice of System and Network Administration"},
    "active directory": {"youtube": "https://youtu.be/85-bp7XxWDQ",  "course": "https://www.udemy.com/course/active-directory-administration-for-helpdesk-technicians", "book": "Active Directory by Brian Desmond"},
    "risk management":  {"youtube": "https://youtu.be/ip7ItZ6D7y4",  "course": "https://www.coursera.org/learn/risk-management",        "book": "Risk Management by Michel Crouhy"},
}

with open(r'C:\CareerShieldAI\datasets\learning_resources.json', 'w') as f:
    json.dump(learning_resources, f, indent=2)

print(f"Learning resources: {len(learning_resources)} skills mapped")
print("Saved to datasets/learning_resources.json")
print("\nSample - Python:")
print(json.dumps(learning_resources['python'], indent=2))

Learning resources: 49 skills mapped
Saved to datasets/learning_resources.json

Sample - Python:
{
  "youtube": "https://youtu.be/rfscVS0vtbw",
  "course": "https://www.coursera.org/learn/python",
  "book": "Automate the Boring Stuff with Python"
}


In [6]:
import sys
import pickle
sys.path.append(r'C:\CareerShieldAI\backend\app')
from sentence_transformers import SentenceTransformer

print("Loading master career dataset...")
master_df = pd.read_csv(r'C:\CareerShieldAI\datasets\master_career_skills.csv')
print(f"Total roles: {len(master_df)}")

print("Loading MiniLM model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Combine job title + skills for better matching
master_df['combined'] = (
    master_df['job_title'].fillna('') + ' ' +
    master_df['skills'].fillna('')
)

print("Generating embeddings — this takes 5-10 minutes...")
embeddings = model.encode(
    master_df['combined'].tolist(),
    show_progress_bar=True,
    batch_size=64
)

# Save new embeddings
with open(r'C:\CareerShieldAI\ml_models\career_embeddings_master.pkl', 'wb') as f:
    pickle.dump(embeddings, f)

print(f"Embeddings shape: {embeddings.shape}")
print("Saved to ml_models/career_embeddings_master.pkl")

Loading master career dataset...
Total roles: 52226
Loading MiniLM model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings — this takes 5-10 minutes...


Batches:   0%|          | 0/817 [00:00<?, ?it/s]

Embeddings shape: (52226, 384)
Saved to ml_models/career_embeddings_master.pkl


In [7]:
# Update career_recommender.py to use master dataset
career_recommender_path = r'C:\CareerShieldAI\backend\app\career_recommender.py'

with open(career_recommender_path, 'r') as f:
    content = f.read()

# Replace old dataset path with new master dataset
content = content.replace(
    r'C:\CareerShieldAI\datasets\career_skills.csv',
    r'C:\CareerShieldAI\datasets\master_career_skills.csv'
)

# Replace old embeddings path with new master embeddings
content = content.replace(
    r'C:\CareerShieldAI\ml_models\career_embeddings.pkl',
    r'C:\CareerShieldAI\ml_models\career_embeddings_master.pkl'
)

with open(career_recommender_path, 'w') as f:
    f.write(content)

print("career_recommender.py updated successfully")
print("Now using master_career_skills.csv (52,226 roles)")
print("Now using career_embeddings_master.pkl")

career_recommender.py updated successfully
Now using master_career_skills.csv (52,226 roles)
Now using career_embeddings_master.pkl


In [8]:
# Update interview_prep.py to include learning resources
interview_prep_path = r'C:\CareerShieldAI\backend\app\interview_prep.py'

new_content = '''import random
import json

# --- Load learning resources ---
with open(r'C:\\CareerShieldAI\\datasets\\learning_resources.json', 'r') as f:
    LEARNING_RESOURCES = json.load(f)

# --- Generate step by step career roadmap with learning resources ---
def generate_roadmap(current_skills, missing_skills, target_role):
    if not missing_skills:
        return ["You already have all skills needed for this role."]

    roadmap = []
    roadmap.append(f"Your goal: {target_role}")

    for i in range(0, len(missing_skills), 2):
        step_num = (i // 2) + 1
        skills_in_step = missing_skills[i:i+2]
        roadmap.append(f"Step {step_num}: Learn {' and '.join(skills_in_step)}")

        # Add learning resources for each skill in this step
        for skill in skills_in_step:
            skill_lower = skill.lower().strip()
            if skill_lower in LEARNING_RESOURCES:
                res = LEARNING_RESOURCES[skill_lower]
                roadmap.append(f"  → YouTube: {res['youtube']}")
                roadmap.append(f"  → Course:  {res['course']}")
                roadmap.append(f"  → Book:    {res['book']}")

    roadmap.append("Final Step: Build 2-3 projects using all learned skills")
    roadmap.append("Final Step: Update resume and apply for roles")
    return roadmap

# --- Get learning resources for a specific skill ---
def get_resources_for_skill(skill):
    skill_lower = skill.lower().strip()
    if skill_lower in LEARNING_RESOURCES:
        return LEARNING_RESOURCES[skill_lower]
    return {"youtube": "https://www.youtube.com/results?search_query=" + skill.replace(" ", "+"),
            "course":  "https://www.coursera.org/search?query=" + skill.replace(" ", "+"),
            "book":    f"Search '{skill}' on Amazon Books"}

# --- Generate technical interview questions ---
def generate_technical_questions(target_role, skills):
    questions = []
    templates = [
        "Explain how you would use {} in a real project.",
        "What are the key concepts in {}?",
        "How does {} work and when would you use it?",
        "What is your experience with {}?",
        "Describe a challenge you faced while working with {}."
    ]
    for skill in skills[:8]:
        template = random.choice(templates)
        questions.append(template.format(skill))
    return questions

# --- Generate HR questions ---
def generate_hr_questions():
    return [
        "Tell me about yourself.",
        "Why do you want to work in this role?",
        "Where do you see yourself in 5 years?",
        "What is your greatest strength and weakness?",
        "Describe a situation where you worked in a team.",
        "How do you handle pressure and tight deadlines?",
        "Why should we hire you over other candidates?"
    ]

# --- Generate project questions ---
def generate_project_questions(skills):
    templates = [
        "Walk me through a project where you used {}.",
        "What was the most challenging project involving {}?",
        "How did you implement {} in one of your projects?",
    ]
    questions = []
    for skill in skills[:5]:
        template = random.choice(templates)
        questions.append(template.format(skill))
    return questions

# --- Master function ---
def generate_interview_kit(target_role, resume_skills, missing_skills):
    return {
        "roadmap":              generate_roadmap(resume_skills, missing_skills, target_role),
        "technical_questions":  generate_technical_questions(target_role, resume_skills),
        "hr_questions":         generate_hr_questions(),
        "project_questions":    generate_project_questions(resume_skills),
        "learning_resources":   {skill: get_resources_for_skill(skill) for skill in missing_skills}
    }

print("Interview prep functions loaded successfully.")
'''

with open(interview_prep_path, 'w') as f:
    f.write(new_content)

print("interview_prep.py updated with learning resources")

UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 995: character maps to <undefined>

In [9]:
with open(interview_prep_path, 'w', encoding='utf-8') as f:
    f.write(new_content)

print("interview_prep.py updated with learning resources")

interview_prep.py updated with learning resources


In [10]:
# Test updated career recommender + interview prep
from career_recommender import recommend_careers, find_skill_gap
from interview_prep import generate_interview_kit

# Test with Python/ML skills
test_skills = ['python', 'machine learning', 'sql', 'aws', 'docker']

print("Testing career recommendation with 52,226 roles...")
recs = recommend_careers(test_skills, top_n=5)

print("\nTop 5 Career Matches:")
for i, rec in enumerate(recs, 1):
    print(f"{i}. {rec['job_title']} ({rec['match_score']}%)")

# Skill gap for top role
top_role = recs[0]
gap = find_skill_gap(test_skills, top_role['required_skills'])

print(f"\nSkill Gap for: {top_role['job_title']}")
print(f"Have:    {gap['matching_skills']}")
print(f"Missing: {gap['missing_skills']}")

# Interview kit with learning resources
kit = generate_interview_kit(
    target_role=top_role['job_title'],
    resume_skills=test_skills,
    missing_skills=gap['missing_skills']
)

print("\nRoadmap with Learning Resources:")
for step in kit['roadmap']:
    print(step)

print("\nLearning Resources for Missing Skills:")
for skill, res in kit['learning_resources'].items():
    print(f"\n{skill}:")
    print(f"  YouTube: {res['youtube']}")
    print(f"  Course:  {res['course']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Career dataset loaded: 52226 roles
Career recommender functions loaded successfully.
Interview prep functions loaded successfully.
Testing career recommendation with 52,226 roles...

Top 5 Career Matches:
1. Python AWS Professional (62.8%)
2. AWS ETL Python Engineer (61.7%)
3. Machine Learning Engineer - Scala/Python (61.5%)
4. SQL DBA (61.1%)
5. Python Engineer for Data Analytics (60.5%)

Skill Gap for: Python AWS Professional
Have:    []
Missing: ['ERPMS SQLLinux', 'Wordpress', 'PLSQLInformatica', 'SSISBusiness intelligence']

Roadmap with Learning Resources:
Your goal: Python AWS Professional
Step 1: Learn ERPMS SQLLinux and Wordpress
Step 2: Learn PLSQLInformatica and SSISBusiness intelligence
Final Step: Build 2-3 projects using all learned skills
Final Step: Update resume and apply for roles

Learning Resources for Missing Skills:

ERPMS SQLLinux:
  YouTube: https://www.youtube.com/results?search_query=ERPMS+SQLLinux
  Course:  https://www.coursera.org/search?query=ERPMS+SQLLinux

In [11]:
import re

# Better skill parser — handles more edge cases
def parse_naukri_skills_v2(skills_text):
    if not isinstance(skills_text, str) or skills_text.strip() == '':
        return ''

    # Split on capital letter following lowercase letter
    skills = re.sub(r'([a-z])([A-Z])', r'\1, \2', skills_text)

    # Split on capital letter following a digit
    skills = re.sub(r'(\d)([A-Z])', r'\1, \2', skills)

    # Split on capital letter following )
    skills = re.sub(r'(\))([A-Z])', r'\1, \2', skills)

    # Remove extra spaces
    skills = re.sub(r'\s+', ' ', skills).strip()

    return skills

# Test the improved parser
test = "ERPMS SQLLinux PLSQLInformatica SSISBusiness intelligence"
print("Before:", test)
print("After: ", parse_naukri_skills_v2(test))

test2 = "Text miningCareer developmentdata scienceFinanceMachine learningSQL"
print("\nBefore:", test2)
print("After: ", parse_naukri_skills_v2(test2))

Before: ERPMS SQLLinux PLSQLInformatica SSISBusiness intelligence
After:  ERPMS SQLLinux PLSQLInformatica SSISBusiness intelligence

Before: Text miningCareer developmentdata scienceFinanceMachine learningSQL
After:  Text mining, Career developmentdata science, Finance, Machine learning, SQL


In [12]:
# Better approach - use known skill keywords to split concatenated skills
KNOWN_SKILLS_SPLIT = [
    'SQL', 'Linux', 'Python', 'Java', 'AWS', 'GCP', 'Azure', 'Docker',
    'Kubernetes', 'Jenkins', 'Terraform', 'Ansible', 'Git', 'Excel',
    'Tableau', 'PowerBI', 'MongoDB', 'MySQL', 'Oracle', 'SAP', 'SSIS',
    'SSRS', 'SSAS', 'Informatica', 'Hadoop', 'Spark', 'Kafka', 'Hive',
    'Scala', 'React', 'Angular', 'Node', 'Django', 'Flask', 'Spring',
    'WordPress', 'Magento', 'Shopify', 'Salesforce', 'ServiceNow',
    'Jira', 'Confluence', 'Selenium', 'Appium', 'Postman', 'Figma',
    'Photoshop', 'Illustrator', 'AutoCAD', 'MATLAB', 'R', 'SAS',
    'SPSS', 'Stata', 'NLP', 'CV', 'ML', 'AI', 'DL', 'API', 'REST',
    'HTML', 'CSS', 'PHP', 'Ruby', 'Swift', 'Kotlin', 'Flutter',
    'Business intelligence', 'Machine learning', 'Deep learning',
    'Data science', 'Data analysis', 'Data engineering',
]

def parse_skills_smart(skills_text):
    if not isinstance(skills_text, str) or skills_text.strip() == '':
        return ''

    # Step 1: Split on lowercase→uppercase boundary
    result = re.sub(r'([a-z])([A-Z])', r'\1, \2', skills_text)

    # Step 2: Split on digit→uppercase boundary
    result = re.sub(r'(\d)([A-Z][a-z])', r'\1, \2', result)

    # Step 3: Clean extra spaces
    result = re.sub(r'\s+', ' ', result).strip()

    # Step 4: Remove very short meaningless tokens (1-2 chars except known ones)
    parts = [p.strip() for p in result.split(',')]
    parts = [p for p in parts if len(p) > 2 or p.upper() in ['R', 'C', 'AI', 'ML', 'DL', 'CV']]

    return ', '.join(parts)

# Rebuild all 3 naukri datasets with better parser
naukri1_v2 = standardize_naukri.__func__(naukri1, 'Job_Titles', 'Skills') if hasattr(standardize_naukri, '__func__') else None

# Redefine standardize with new parser
def standardize_naukri_v2(df, title_col, skills_col):
    result = pd.DataFrame()
    result['job_title'] = df[title_col].fillna('')
    result['skills']    = df[skills_col].apply(parse_skills_smart)
    result['source']    = 'naukri'
    return result[result['job_title'] != '']

naukri1_v2 = standardize_naukri_v2(naukri1, 'Job_Titles', 'Skills')
naukri2_v2 = standardize_naukri_v2(naukri2, 'Job Titles', 'Skills')
naukri3_v2 = standardize_naukri_v2(naukri3, 'Job_Titles', 'Skills')

# Rebuild master dataset
master_v2 = pd.concat([
    career_df[['job_title', 'skills', 'source']],
    naukri1_v2,
    naukri2_v2,
    naukri3_v2
], ignore_index=True)

master_v2 = master_v2[master_v2['skills'].str.strip() != '']
master_v2.drop_duplicates(subset=['job_title', 'skills'], inplace=True)
master_v2.reset_index(drop=True, inplace=True)

# Save
master_v2.to_csv(r'C:\CareerShieldAI\datasets\master_career_skills.csv', index=False)
print(f"Master dataset rebuilt: {len(master_v2)} roles")

# Test sample
print("\nSample skills after fix:")
sample = master_v2[master_v2['source']=='naukri']['skills'].iloc[5]
print(sample[:200])

Master dataset rebuilt: 52225 roles

Sample skills after fix:
deep learningdata science, GCPNeural networks, Machine learning, Cloud, Data processing, Subject Matter Expert


In [13]:
# Final approach - comprehensive skill splitter
def parse_skills_final(skills_text):
    if not isinstance(skills_text, str) or skills_text.strip() == '':
        return ''

    text = skills_text

    # Step 1: lowercase→uppercase split
    text = re.sub(r'([a-z])([A-Z])', r'\1, \2', text)

    # Step 2: letter→digit split  
    text = re.sub(r'([a-zA-Z])(\d)', r'\1, \2', text)

    # Step 3: digit→letter split
    text = re.sub(r'(\d)([a-zA-Z])', r'\1, \2', text)

    # Step 4: Fix known stuck patterns manually
    stuck_pairs = [
        ('data science', 'machine'), ('data science', 'deep'),
        ('data science', 'python'), ('data science', 'sql'),
        ('machine learning', 'data'), ('machine learning', 'deep'),
        ('deep learning', 'data'), ('deep learning', 'machine'),
        ('neural networks', 'machine'), ('neural networks', 'deep'),
        ('cloud', 'data'), ('cloud', 'machine'), ('cloud', 'deep'),
        ('gcp', 'neural'), ('gcp', 'machine'), ('gcp', 'data'),
        ('aws', 'machine'), ('aws', 'data'), ('aws', 'deep'),
        ('azure', 'machine'), ('azure', 'data'), ('azure', 'deep'),
    ]

    text_lower = text.lower()
    for pair in stuck_pairs:
        pattern = pair[0] + pair[1]
        replacement = pair[0] + ', ' + pair[1]
        text_lower = text_lower.replace(pattern, replacement)
    text = text_lower

    # Step 5: Clean up
    text = re.sub(r'\s+', ' ', text).strip()
    parts = [p.strip() for p in text.split(',') if len(p.strip()) > 1]

    return ', '.join(parts)

# Test
print("Test 1:", parse_skills_final("deep learningdata scienceGCPNeural networks"))
print("Test 2:", parse_skills_final("Text miningCareer developmentMachine learningSQL"))
print("Test 3:", parse_skills_final("AWSMachine learningData sciencePython"))

Test 1: deep learning, data science, gcp, neural networks
Test 2: text mining, career development, machine learning, sql
Test 3: aws, machine learning, data science, python


In [14]:
# Rebuild master dataset with final parser
def standardize_final(df, title_col, skills_col):
    result = pd.DataFrame()
    result['job_title'] = df[title_col].fillna('')
    result['skills']    = df[skills_col].apply(parse_skills_final)
    result['source']    = 'naukri'
    return result[result['job_title'] != '']

naukri1_final = standardize_final(naukri1, 'Job_Titles', 'Skills')
naukri2_final = standardize_final(naukri2, 'Job Titles', 'Skills')
naukri3_final = standardize_final(naukri3, 'Job_Titles', 'Skills')

# Add original career data
career_df['source'] = 'original'

master_final = pd.concat([
    career_df[['job_title', 'skills', 'source']],
    naukri1_final,
    naukri2_final,
    naukri3_final
], ignore_index=True)

# Clean
master_final = master_final[master_final['skills'].str.strip() != '']
master_final.drop_duplicates(subset=['job_title', 'skills'], inplace=True)
master_final.reset_index(drop=True, inplace=True)

# Save
master_final.to_csv(
    r'C:\CareerShieldAI\datasets\master_career_skills.csv', index=False)

print(f"Final master dataset: {len(master_final)} roles")
print(f"\nSample skills (fixed):")
for i in range(3):
    sample = master_final[master_final['source']=='naukri']['skills'].iloc[i]
    print(f"{i+1}. {sample[:150]}")

Final master dataset: 52225 roles

Sample skills (fixed):
1. text mining, career developmentdata science, finance, machine learning, data mining, stakeholder management, sql
2. advanced statistical analysis, data governance and quality assurance, machine learning and aidata analysis and visualization, management, quality assu
3. product management, career development, operations research, data modeling, analytical, relationship building, machine learning, sql


In [15]:
# Rebuild embeddings with final master dataset
print("Regenerating embeddings with final master dataset...")

embeddings_final = model.encode(
    master_final['combined'].tolist() if 'combined' in master_final.columns
    else (master_final['job_title'].fillna('') + ' ' + master_final['skills'].fillna('') ).tolist(),
    show_progress_bar=True,
    batch_size=64
)

with open(r'C:\CareerShieldAI\ml_models\career_embeddings_master.pkl', 'wb') as f:
    pickle.dump(embeddings_final, f)

print(f"Embeddings shape: {embeddings_final.shape}")
print("Saved to ml_models/career_embeddings_master.pkl")
print("\nAll done. Summary:")
print(f"master_career_skills.csv  → {len(master_final)} roles")
print(f"salary_demand.csv         → 67 roles with Indian salaries")
print(f"learning_resources.json   → 49 skills with resources")
print(f"career_embeddings_master  → {embeddings_final.shape} embeddings")

Regenerating embeddings with final master dataset...


Batches:   0%|          | 0/817 [00:00<?, ?it/s]

Embeddings shape: (52225, 384)
Saved to ml_models/career_embeddings_master.pkl

All done. Summary:
master_career_skills.csv  → 52225 roles
salary_demand.csv         → 67 roles with Indian salaries
learning_resources.json   → 49 skills with resources
career_embeddings_master  → (52225, 384) embeddings
